In [2]:
import numpy as np
import scipy.stats as sps
from sklearn.datasets import load_iris
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C

In [3]:
def objective(params):    
    return sum(params)

In [4]:
# 创建形状为 (6, 2) 的 NumPy 数组 bounds
bounds = np.array([[1, 100], [1, 100], [1, 100], [1, 100], [1, 100], [1, 100]])

In [5]:
X = np.random.uniform(low=bounds[:, 0], high=bounds[:, 1], size=(20, 6))
print(X)

[[88.25851897 98.01527269 98.3770979  21.58675353 67.27460148  4.4419017 ]
 [83.16667857 74.58908307 47.20448607 90.93784777 15.72052426 85.2498768 ]
 [59.97321678 50.97000192 61.83317755 37.10758879 85.51376311 82.35751841]
 [ 3.47379777 15.00426521 72.50662799 17.19295387 77.04305127 46.70258711]
 [29.31942066 81.61210648 38.06204804 11.89006525 30.61008675 25.03780947]
 [92.49047595 73.5447679  72.33563847 49.11423625 62.63714308 46.49668352]
 [28.55086218 85.43216052 45.40650438 97.02623387 56.55141084 89.46100841]
 [51.11477762 73.54489928 56.90600658 42.52006121 12.17078948 70.79724465]
 [85.13657724 58.48245863 70.40458638 80.40199432 54.67664473 67.60350312]
 [90.39527747 59.79043964 50.89673871 56.3473715  52.87225163 77.43085533]
 [56.71449059 80.08315093 88.29611852 58.81655448 83.86876263 10.56829666]
 [ 9.03290633 34.98488907 52.84848763 42.12625386  9.3657649  27.0599096 ]
 [39.31539522 82.245872   81.96598043 95.63523812 75.13574185 94.73850972]
 [91.4606352  10.55460596

In [6]:
y = np.random.uniform(1, 600, size=X.shape[0])  # Generate one value per row in X
print(y)

[188.98043409 395.91555584 329.02256138 572.36436705 458.1971112
 531.78277351  25.2959376  554.45492894 276.33240038 137.25399381
 490.75724167 176.17920887 161.38476282  92.54516026 410.64057126
  62.17070211  91.41534191  62.67842111  47.03290808  19.02526001]


In [7]:
# 定义核函数
kernel = C(1.0, (1e-3, 1e3)) * RBF(1.0, (1e-2, 1e2))

# 创建高斯过程回归模型
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)

# # 拟合模型
# gp.fit(X, y)

In [8]:
def acquisition(x):
    mu, sigma = gp.predict(x.reshape(1, -1), return_std=True)
    f_best = np.max(y)
    improvement = f_best - mu
    with np.errstate(divide='warn'):
        Z = improvement / sigma if sigma > 0 else 0
        ei = improvement * sps.norm.cdf(Z) + sigma * sps.norm.pdf(Z)
        ei[sigma == 0.0] == 0.0
    return ei

In [9]:
n_random_points = 10000
x_random_points = np.random.uniform(bounds[:, 0], bounds[:, 1], size=(n_random_points, bounds.shape[0]))
print(x_random_points)

[[85.67325054 31.23257073 64.7511774  42.02807    47.50106798 16.14464262]
 [38.0536106   9.86688838  1.99051758 46.47311248 19.8692869  42.6885086 ]
 [33.58913605 10.650995   52.41262789 90.23390752 33.72815735 85.67278553]
 ...
 [35.89321841 28.77211241 55.07667926 88.62882886 99.97702629 27.99771185]
 [48.44508457 29.68197062  7.6542771  78.47279463 65.71035539  7.17333991]
 [33.06814669 57.3523521  12.31463167 89.46700734 47.29792161 92.67964042]]


In [10]:
# 运行 n_iter 次的贝叶斯优化循环
n_iter = 10
for i in range(n_iter):
    # 使用现有样本更新高斯过程
    gp.fit(X, y)

    # 通过优化获取函数找到下一个样本
    x_next = None
    best_acq_value = -np.inf

    # 从参数空间中抽样大量随机点
    n_random_points = 10000
    x_random_points = np.random.uniform(bounds[:, 0], bounds[:, 1], size=(n_random_points, bounds.shape[0]))

    # 在每个点上评估获取函数并找到最大值
    acq_values = np.array([acquisition(x) for x in x_random_points])
    max_acq_index = np.argmax(acq_values)
    max_acq_value = acq_values[max_acq_index]

    if max_acq_value > best_acq_value:
        best_acq_value = max_acq_value
        x_next = x_random_points[max_acq_index]

    print(f"Iteration {i+1}: next sample is {x_next}, acquisition value is {best_acq_value}")

    # 在下一个样本上评估目标函数并将其添加到现有样本中
    y_next = objective(x_next)
    X = np.vstack((X, x_next))
    y = np.append(y, y_next)

c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Iteration 1: next sample is [77.96207184 93.25115775 21.08227464  4.54659754 31.32397697  8.78047701], acquisition value is [648.83636538]


c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Iteration 2: next sample is [93.31903063 56.99697663 20.04819607 34.27154137 46.07293318 81.15927142], acquisition value is [583.19761764]


c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


...

Iteration 8: next sample is [90.77778359 89.22701701 74.0330234   6.74591975 62.21718161 31.12996162], acquisition value is [576.64729721]


c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Iteration 9: next sample is [ 2.50683452 79.88338781  3.94145325 56.35018228 31.22438506 81.63843345], acquisition value is [572.36436705]


c:\softwares\develop\anaconda3\Lib\site-packages\sklearn\gaussian_process\kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Iteration 10: next sample is [99.60267143 64.49513661 95.71271021 17.41724746 95.8777922  16.52307556], acquisition value is [574.55558854]
